# Notebook 3: Structured Extraction Accuracy

**Purpose:** Measure F1 accuracy of board state extraction for models that passed Notebook 2.
To swap in a different model, change `MODEL_IDS_TO_TEST` in Cell 2.

**F1 definitions:**
- **Piece presence F1:** does the predicted `pieces` array contain an item with the correct `square_id`?
  (ignores what piece_type was predicted)
- **Piece identity F1:** does the predicted array have the correct `square_id` AND correct `piece_type`?
- **Misidentification counts as:** presence TP + identity FN. This separates detection from
  classification — a model that finds the right squares but misidentifies pieces scores well on
  presence but poorly on identity.

**Winner definition:** must satisfy ALL of:
- p95 < 800ms (Notebook 2)
- F1 presence > 0.85
- F1 identity > 0.75

In [ ]:
# ---- CELL 1: Environment check ----
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '../requirements.txt'])

import pkg_resources
print('=== Environment ===')
for pkg in ['openai', 'Pillow', 'scikit-learn', 'numpy', 'tqdm']:
    try:
        print(f'  {pkg}: {pkg_resources.get_distribution(pkg).version}')
    except Exception:
        print(f'  {pkg}: NOT FOUND')
try:
    import vllm; print(f'  vllm: {vllm.__version__}')
except ImportError:
    pass
print('Done.')

In [ ]:
# ---- CELL 2: Configuration ----
# MODEL_IDS_TO_TEST: copy the passing models from Notebook 2's output.

from pathlib import Path

# Models to test accuracy for (must have passed Notebook 2 latency gate)
MODEL_IDS_TO_TEST = {
    'qwen25vl_7b_awq': 'Qwen/Qwen2.5-VL-7B-Instruct-AWQ',
    # Add other passing models here:
    # 'cosmos_nemotron': 'nvidia/Cosmos-Nemotron-34B-Vision',
}

VLLM_HOST = 'localhost'
VLLM_PORT = 8000
VLLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'

IMAGES_DIR = Path('../data/images')
GT_DIR     = Path('../data/ground_truth')

# Accuracy thresholds
F1_PRESENCE_THRESHOLD = 0.85
F1_IDENTITY_THRESHOLD = 0.75

SWAP_MEMORY_TIMEOUT_S  = 1200
VLLM_STARTUP_TIMEOUT_S = 300

print(f'Models to test: {list(MODEL_IDS_TO_TEST.keys())}')
print(f'F1 presence threshold: {F1_PRESENCE_THRESHOLD}')
print(f'F1 identity threshold: {F1_IDENTITY_THRESHOLD}')

In [ ]:
# ---- CELL 3: Schema and helpers ----

import base64, io, json, time, requests
import numpy as np
from PIL import Image
from openai import OpenAI

PIECE_TYPE_ENUM = ['house', 'hotel', 'token_red', 'token_blue', 'token_green', 'token_yellow']
# Add other token colors your physical set uses

BOARD_STATE_SCHEMA = {
    'type': 'object',
    'properties': {
        'pieces': {
            'type': 'array',
            'items': {
                'type': 'object',
                'properties': {
                    'square_id':  {'type': 'string', 'pattern': '^[0-3][0-9]$'},
                    'piece_type': {'type': 'string', 'enum': PIECE_TYPE_ENUM},
                    'owner':      {'type': 'string'},
                },
                'required': ['square_id', 'piece_type', 'owner'],
                'additionalProperties': False,
            },
        }
    },
    'required': ['pieces'],
    'additionalProperties': False,
}

EXTRACTION_PROMPT = (
    'Look at this Monopoly board image. '
    'Return a JSON object listing every occupied square. '
    'Square IDs are zero-padded two-digit numbers: 00=Go, 01=Mediterranean Ave, '
    '...39=Boardwalk. '
    'Include only squares that have a piece or building on them. '
    'For each piece, specify square_id, piece_type, and owner.'
)

VALID_SQUARE_IDS = {f'{i:02d}' for i in range(40)}

def image_to_b64(image_path: Path, max_size: int = 1024) -> str:
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    if max(w, h) > max_size:
        scale = max_size / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=90)
    return base64.b64encode(buf.getvalue()).decode('utf-8')

def make_vision_message(b64_image: str, prompt: str) -> list:
    return [{'role': 'user', 'content': [
        {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64_image}'}},
        {'type': 'text', 'text': prompt},
    ]}]

def wait_for_vllm(url, timeout_s=300, poll_s=5):
    health_url = url.replace('/v1', '/health')
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            if requests.get(health_url, timeout=2).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(poll_s)
    return False

def wait_for_gpu_clear(timeout_s=SWAP_MEMORY_TIMEOUT_S, threshold_mb=2048):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                text=True
            ).strip()
            if int(out.split('\n')[0]) < threshold_mb:
                return True
        except Exception:
            return True
        time.sleep(5)
    return False

print('Helpers loaded.')

In [ ]:
# ---- CELL 4: Ground truth integrity check ----
# Validates all GT files before the F1 loop starts.
# Fails loudly on any malformed file — do not run F1 with corrupt denominators.

def load_and_validate_gt(gt_dir: Path) -> dict:
    """Load all *_gt.json files with full integrity checks.
    Returns {image_stem: [{square_id, piece_type, owner}]}.
    Raises ValueError with descriptive message on any issue.
    """
    gt_data = {}
    errors = []

    for gt_file in sorted(gt_dir.glob('*_gt.json')):
        try:
            with open(gt_file) as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            errors.append(f'{gt_file.name}: invalid JSON — {e}')
            continue

        pieces = data.get('pieces', [])
        if not isinstance(pieces, list):
            errors.append(f'{gt_file.name}: "pieces" must be a list')
            continue

        seen_ids = set()
        for i, p in enumerate(pieces):
            sid = p.get('square_id', '')
            pt  = p.get('piece_type', '')

            if sid not in VALID_SQUARE_IDS:
                errors.append(f'{gt_file.name}[{i}]: invalid square_id "{sid}" (must be "00"–"39")')
            if sid in seen_ids:
                errors.append(f'{gt_file.name}[{i}]: duplicate square_id "{sid}"')
            seen_ids.add(sid)

            if pt not in PIECE_TYPE_ENUM:
                errors.append(
                    f'{gt_file.name}[{i}]: unknown piece_type "{pt}".\n'
                    f'  Valid types: {PIECE_TYPE_ENUM}\n'
                    f'  To add a new type, update PIECE_TYPE_ENUM in Cell 3 and BOARD_STATE_SCHEMA.'
                )

        image_stem = gt_file.stem.replace('_gt', '')
        gt_data[image_stem] = pieces

    if errors:
        raise ValueError('Ground truth integrity check FAILED:\n' + '\n'.join(errors))

    return gt_data

print('Loading and validating ground truth files...')
gt_data = load_and_validate_gt(GT_DIR)

total_pieces = sum(len(v) for v in gt_data.values())
print(f'Ground truth files: {len(gt_data)}')
print(f'Total occupied squares: {total_pieces}')
print('Ground truth integrity check PASSED.')

if len(gt_data) < 20:
    print(f'\nWARNING: only {len(gt_data)} GT files found. Target is 30.')
    print('F1 scores will be noisier with fewer images.')

In [ ]:
# ---- CELL 5: F1 computation ----

def compute_f1(tp: int, fp: int, fn: int) -> float:
    if tp + fp + fn == 0:
        return 1.0  # no pieces in image and none predicted = perfect
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def evaluate_prediction(gt_pieces: list, pred_pieces: list) -> dict:
    """
    F1 scoring:
      Presence: correct square_id in prediction (regardless of piece_type)
      Identity: correct square_id AND correct piece_type

    Misidentification (right square, wrong piece_type):
      → presence TP, identity FN
      (separates detection from classification)
    """
    # Filter invalid square_ids from predictions
    pred_pieces = [p for p in pred_pieces if p.get('square_id', '') in VALID_SQUARE_IDS]

    gt_by_id   = {p['square_id']: p for p in gt_pieces}
    pred_by_id = {p['square_id']: p for p in pred_pieces}

    gt_ids   = set(gt_by_id.keys())
    pred_ids = set(pred_by_id.keys())

    # Presence
    pres_tp = len(gt_ids & pred_ids)
    pres_fp = len(pred_ids - gt_ids)
    pres_fn = len(gt_ids - pred_ids)

    # Identity: correct square AND correct piece_type
    ident_tp = sum(
        1 for sid in (gt_ids & pred_ids)
        if gt_by_id[sid]['piece_type'] == pred_by_id[sid]['piece_type']
    )
    ident_fp = pres_fp  # hallucinated squares = identity FP
    ident_fn = pres_fn + (pres_tp - ident_tp)  # missed + misidentified

    return {
        'presence_f1': compute_f1(pres_tp, pres_fp, pres_fn),
        'identity_f1': compute_f1(ident_tp, ident_fp, ident_fn),
        'pres_tp': pres_tp, 'pres_fp': pres_fp, 'pres_fn': pres_fn,
        'ident_tp': ident_tp, 'ident_fp': ident_fp, 'ident_fn': ident_fn,
    }

print('F1 functions ready.')

In [ ]:
# ---- CELL 6: Model swap (same as Notebook 2) ----

current_proc = None

def build_vllm_cmd(model_id: str) -> list:
    cmd = [
        sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
        '--model', model_id,
        '--skip-mm-profiling',
        '--attention-backend', 'TRITON_ATTN',
        '--dtype', 'auto',
        '--host', VLLM_HOST,
        '--port', str(VLLM_PORT),
    ]
    if 'vila' in model_id.lower():
        cmd.append('--trust-remote-code')
    return cmd

def swap_model(model_id: str):
    global current_proc
    if current_proc is not None and current_proc.poll() is None:
        current_proc.terminate()
        try:
            current_proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            current_proc.kill()
        current_proc = None
        print('Waiting for GPU memory...')
        wait_for_gpu_clear()
        print('GPU cleared.')

    proc = subprocess.Popen(
        build_vllm_cmd(model_id),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if not wait_for_vllm(VLLM_BASE_URL, timeout_s=VLLM_STARTUP_TIMEOUT_S):
        proc.terminate()
        return None
    current_proc = proc
    return proc

print('swap_model() ready.')

In [ ]:
# ---- CELL 7: Main accuracy benchmark loop ----

from tqdm import tqdm

accuracy_results = {}

for model_key, model_id in MODEL_IDS_TO_TEST.items():
    print(f'\n{"="*60}')
    print(f'Model: {model_key} ({model_id})')
    print(f'{'='*60}')

    proc = swap_model(model_id)
    if proc is None:
        print(f'SKIPPED: {model_key} failed to load.')
        accuracy_results[model_key] = {'status': 'load_failed'}
        continue

    client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-required')

    per_image = []
    errors = []

    for image_stem, gt_pieces in tqdm(gt_data.items(), desc=model_key):
        img_candidates = list(IMAGES_DIR.glob(f'{image_stem}.*'))
        if not img_candidates:
            print(f'  WARNING: no image for {image_stem} — skipping')
            continue

        b64 = image_to_b64(img_candidates[0])
        messages = make_vision_message(b64, EXTRACTION_PROMPT)

        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=messages,
                max_tokens=512,
                temperature=0.0,
                extra_body={'guided_json': BOARD_STATE_SCHEMA},
            )
            text = resp.choices[0].message.content
            parsed = json.loads(text)
            pred_pieces = parsed.get('pieces', [])
        except (json.JSONDecodeError, Exception) as e:
            errors.append({'image': image_stem, 'error': str(e)})
            pred_pieces = []

        scores = evaluate_prediction(gt_pieces, pred_pieces)
        per_image.append({'image': image_stem, **scores})

    # Aggregate F1 across all images
    total_pres_tp = sum(r['pres_tp'] for r in per_image)
    total_pres_fp = sum(r['pres_fp'] for r in per_image)
    total_pres_fn = sum(r['pres_fn'] for r in per_image)
    total_ident_tp = sum(r['ident_tp'] for r in per_image)
    total_ident_fp = sum(r['ident_fp'] for r in per_image)
    total_ident_fn = sum(r['ident_fn'] for r in per_image)

    overall_presence_f1 = compute_f1(total_pres_tp, total_pres_fp, total_pres_fn)
    overall_identity_f1 = compute_f1(total_ident_tp, total_ident_fp, total_ident_fn)

    accuracy_results[model_key] = {
        'status': 'ok',
        'model_id': model_id,
        'n_images': len(per_image),
        'n_errors': len(errors),
        'presence_f1': overall_presence_f1,
        'identity_f1': overall_identity_f1,
        'per_image': per_image,
        'errors': errors,
    }

    print(f'\n  Images tested:   {len(per_image)}')
    print(f'  Parse errors:    {len(errors)}')
    print(f'  Presence F1:     {overall_presence_f1:.3f} (threshold: {F1_PRESENCE_THRESHOLD})')
    print(f'  Identity F1:     {overall_identity_f1:.3f} (threshold: {F1_IDENTITY_THRESHOLD})')

if current_proc is not None and current_proc.poll() is None:
    current_proc.terminate()
    current_proc.wait()
    print('\nvLLM stopped.')

In [ ]:
# ---- CELL 8: Results table + winner selection ----

print(f'{'Model':<22} {'Images':>7} {'Errors':>7} {'Pres F1':>9} {'Ident F1':>9}  Verdict')
print('-' * 75)

winner = None
winners = []

for model_key, res in accuracy_results.items():
    if res.get('status') == 'load_failed':
        print(f'{model_key:<22} LOAD FAILED')
        continue

    pf1 = res['presence_f1']
    if1 = res['identity_f1']
    n   = res['n_images']
    e   = res['n_errors']

    pres_pass  = pf1 >= F1_PRESENCE_THRESHOLD
    ident_pass = if1 >= F1_IDENTITY_THRESHOLD

    if pres_pass and ident_pass:
        verdict = 'PASS'
        winners.append(model_key)
    elif pres_pass:
        verdict = 'FAIL (identity)'
    elif ident_pass:
        verdict = 'FAIL (presence)'
    else:
        verdict = 'FAIL'

    print(f'{model_key:<22} {n:>7} {e:>7} {pf1:>9.3f} {if1:>9.3f}  {verdict}')

print('-' * 75)

print()
if winners:
    print(f'PASSING MODELS: {winners}')
    print()
    print('These models satisfy both F1 thresholds.')
    print('Cross-reference with Notebook 2 latency results to confirm the joint winner:')
    print(f'  Winner must also have p95 < 800ms from Notebook 2.')
    print()
    # Best model = highest identity F1 among passing
    best = max(winners, key=lambda k: accuracy_results[k]['identity_f1'])
    print(f'RECOMMENDED WINNER: {best}')
    print(f'  Presence F1: {accuracy_results[best]["presence_f1"]:.3f}')
    print(f'  Identity F1: {accuracy_results[best]["identity_f1"]:.3f}')
    print()
    print(f'Next step: convert {accuracy_results[best]["model_id"]} to TensorRT Edge-LLM.')
else:
    print('NO MODELS PASSED accuracy thresholds.')
    print('Consider:')
    print('  - Checking if the camera angle is clear enough (revisit occlusion gate)')
    print('  - Trying a larger model that passed latency marginally')
    print('  - Adjusting the extraction prompt for better structured output')

In [ ]:
# ---- CELL 9: Save results ----

import json
from datetime import datetime

def serialize(obj):
    if isinstance(obj, float):
        return round(obj, 4)
    if isinstance(obj, dict):
        return {k: serialize(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [serialize(x) for x in obj]
    return obj

out_path = Path('../data') / f'accuracy_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
with open(out_path, 'w') as f:
    json.dump({
        'timestamp': datetime.now().isoformat(),
        'config': {
            'f1_presence_threshold': F1_PRESENCE_THRESHOLD,
            'f1_identity_threshold': F1_IDENTITY_THRESHOLD,
            'n_images': len(gt_data),
        },
        'results': serialize(accuracy_results),
        'winners': winners,
    }, f, indent=2)

print(f'Results saved to {out_path}')

In [ ]:
# ---- CELL 10: Failure analysis (optional) ----
# Run this cell to see which specific images caused the most FNs and FPs.
# Useful for diagnosing whether failures are camera angle, model, or prompt issues.

MODEL_KEY_TO_INSPECT = list(accuracy_results.keys())[0]  # change as needed

res = accuracy_results.get(MODEL_KEY_TO_INSPECT, {})
per_image = res.get('per_image', [])

if not per_image:
    print('No per-image data.')
else:
    print(f'Worst images for {MODEL_KEY_TO_INSPECT} (by presence FN):')
    worst = sorted(per_image, key=lambda r: r['pres_fn'], reverse=True)[:10]
    print(f'  {'Image':<25} {'Pres F1':>8} {'Ident F1':>9} {'FN':>4} {'FP':>4}')
    print('  ' + '-' * 55)
    for r in worst:
        print(f'  {r["image"]:<25} {r["presence_f1"]:>8.3f} {r["identity_f1"]:>9.3f} '
              f'{r["pres_fn"]:>4} {r["pres_fp"]:>4}')